In [0]:
from pyspark.sql import SparkSession
# Create a SparkSession if one doesn't exist
spark = SparkSession.builder.appName("CreateDataFrameExample").getOrCreate()

#this is basic comment only.

data = [
    (1, "TV", "Electronics", 1000, "2024-01-05", "Mumbai"),
    (2, "Mobile", "Electronics", 500, "2024-01-07", "Delhi"),
    (3, "Shoes", "Fashion", 150, "2024-02-01", "Mumbai"),
    (4, "Laptop", "Electronics", 1200, "2024-02-10", "Bangalore"),
    (5, "Shirt", "Fashion", 80, "2024-02-15", "Delhi"),
    (6, "TV", "Electronics", 1100, "2024-03-01", "Mumbai")
]

columns = ["order_id", "product", "category", "amount", "order_date", "city"]

df = spark.createDataFrame(data, schema = columns)

df.show()

In [0]:
# 1. Add a new column called amount_with_tax where tax is 18%.

df = df.withColumn("amount_with_tax", df.amount % 18)

df.show(5)

In [0]:
# 2. Create a column high_value:
#"Yes" if amount > 1000
#"No" otherwise

from pyspark.sql.functions import when
from pyspark.sql.functions import col

df = df.withColumn("high_value", when(col("amount") > 1000 , "Yes").otherwise("No"))

df.show(5)

In [0]:
%skip
# 3. Convert order_date from string to date type.

from pyspark.sql.functions import to_date


df = df.withColumn(
    "order_date",
    to_date("order_date", "dd-MM-yyyy")
)

df.show(5)


In [0]:
#4. Get all Electronics orders from Mumbai.
from pyspark.sql.functions import col

df_filter = df.filter(
    (col("city") == "Mumbai") & (col("category") == "Electronics")
)

df_filter.show()

In [0]:
# 5. Find orders between 500 and 1100 amount.

#df.show()

df_between = df.filter(col("amount").between(500,1100))

df_between.show()

In [0]:
#6. Find total sales per category.

#df.show()

from pyspark.sql.functions import sum

df_group = df.groupBy("category").agg(sum("amount").alias("total_sales"))

df_group.show()

In [0]:
# 7. Find average sales per city.

#df.show()

from pyspark.sql.functions import avg

df_avg = df.groupBy("city").agg(avg("amount")).alias("avg_sales")

df_avg.show()

In [0]:
# 8. Rank products by amount within each category.

#df.show()

from pyspark.sql.window import Window
from pyspark.sql.functions import rank

window_spec = Window.partitionBy("category").orderBy(col("amount").desc())
df_rnk_prd = df.withColumn("rank",rank().over(window_spec))

df_rnk_prd = df_rnk_prd.filter(col("rank") == 1)
df_rnk_prd.show()

In [0]:
# 9. Find running total of sales per category ordered by date.

#df.show()

from pyspark.sql.window import Window
from pyspark.sql.functions import sum, row_number

window_spec = Window.partitionBy("category").orderBy("order_date")

df_run_totl = df.withColumn("running_total", sum("amount").over(window_spec))

df_run_totl.select("category","amount","running_total","order_date").show()

In [0]:
# 10. Find previous order amount (lag function).

from pyspark.sql.functions import lag

window_spec = Window.partitionBy("category").orderBy("order_date")

df_prev_order = df.withColumn(
    "previous_amount", lag("amount",1).over(window_spec)
)

df_prev_order.show()

In [0]:
# 11. Find dense rank of products by amount within each category.

#df.show()

from pyspark.sql.window import Window
from pyspark.sql.functions import dense_rank, col
window_spec = Window.partitionBy("category").orderBy(col("amount"))

df_dense_rnk = df.withColumn("dense_rank", dense_rank().over(window_spec))

#df_dense_rnk = df_dense_rnk.filter(col("dense_rank") == 1)

df_dense_rnk.show()

In [0]:
# 12.Return only 2nd highest sale per category.

from pyspark.sql.functions import dense_rank, col
from pyspark.sql.window import Window

window_spec = Window.partitionBy("category").orderBy(col("amount"))

df_2nd_sal = df.withColumn("dense_rank",dense_rank().over(window_spec))

df_2nd_sal = df_2nd_sal.filter(col("dense_rank") == 2)

df_2nd_sal.show()

In [0]:
# 12.Return only 2nd highest sale per category.
#using row_number

from pyspark.sql.functions import row_number, col
from pyspark.sql.window import Window

window_spec = Window.partitionBy("category").orderBy(col("amount"))

df_2nd_sal = df.withColumn("row_number", row_number().over(window_spec))

df_2nd_sal = df_2nd_sal.filter(col("row_number") == 2)

df_2nd_sal.show()

In [0]:
# 13. Add column showing % contribution of each product in its category.
#Formula conceptually:
#percentage=(products​ales/totalc​ategorys​ales)∗100

#df.show()

from pyspark.sql.functions import col, sum, lit, round

df_grouped = df.groupBy("category").agg(sum(col("amount")).alias("total_cat_Sales"))

#df_grouped.show()

df_joined = df.join(df_grouped, df.category == df_grouped.category, 'inner')

df_joined = df_joined.withColumn("percent_contri",round(col("amount")/ col("total_cat_Sales")*100,3))

df_joined.show()




In [0]:
# 14. Add column showing difference between current and previous order amount per category.

#df.show()

from pyspark.sql.functions import lag
from pyspark.sql.functions import window

window_spec = Window.partitionBy("category").orderBy("amount")

df_diff = df.withColumn("prev_amount", lag("amount",1).over(window_spec))

df_diff = df_diff.withColumn("diff", col("amount") - col("prev_amount"))

df_diff.show()


In [0]:
# 15. Calculate moving average of last 2 rows per category ordered by date.

#df.show()

from pyspark.sql.functions import avg
from pyspark.sql.functions import window

window_spec = Window.partitionBy("category").orderBy("order_date").rowsBetween(-2,0)

df_avg = df.withColumn("mov_avg", avg("amount").over(window_spec))

df_avg.show()




In [0]:
# 16. Find first order per city based on date.

#df.show()

from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

window_spec = Window.partitionBy("city").orderBy("order_date")

df_1st_order = df.withColumn("row_number", row_number().over(window_spec))

df_1st_order = df_1st_order.filter(col("row_number") == 1)

df_1st_order.show()


In [0]:
# 17. Add column showing total number of orders per category.

from pyspark.sql.functions import window
from pyspark.sql.functions import count

window_spec = Window.partitionBy("category")

df_tot_order = df.withColumn("total_order_cat", count(col("order_id")).over(window_spec))

df_tot_order.show()



In [0]:
# 18. Find Cities Where Electronics > Fashion
#Step 1: Aggregate
#Step 2: Compare

#df.show()

from pyspark.sql.functions import sum

df_pivot = df.groupBy("city").pivot("category").agg(sum("amount"))

df_pivot = df_pivot.filter(col("Electronics") > col("Fashion"))

df_pivot.show()



In [0]:
# 19. Remove Duplicate Records (Keep Latest)
#Assume duplicate order_id exists. Keep latest order by date.

#df.show()

from pyspark.sql.functions import row_number
from pyspark.sql.functions import window

window_Spec = Window.partitionBy("order_id").orderBy(col("order_date").desc())

df_latest_order = df.withColumn("row_number", row_number().over(window_Spec))

df_latest_order.filter(col("row_number")== 1).show()


In [0]:
# 20. Month-over-Month Growth
#Step 1: Extract month
#Step 2: Aggregate per month
#Step 3: Use lag
#Step 4: Calculate growth
#Growth formula:
# growth=((currentm​onth−previousm​onth)/previousm​onth)∗100

#df.show()

from pyspark.sql.functions import month, sum
from pyspark.sql.functions import window

df_month = df.withColumn("month",month("order_date"))

df_month = df_month.groupBy("month").agg(sum("amount").alias("total_sales"))

window_spec = Window.orderBy("month")

df_month = df_month.withColumn(
    "prev_month_sales",
    lag("total_sales").over(window_spec)
).withColumn(
    "growth_percent",
    ((col("total_sales") - col("prev_month_sales")) / col("prev_month_sales")) * 100
)

df_month.show()
